In [ ]:
import BrukerPS
import KepcoPS
import SR850LIA
import numpy as np
from BrukerPS import BrukerPS
from KepcoPS import KepcoPS
import SR850LIA as SR850
import time

from pymeasure.experiment import Parameter, FloatParameter, IntegerParameter
from pymeasure.experiment import Procedure
from pymeasure.experiment import Results, Worker

from pymeasure.experiment import Experiment




class AngularScan(Procedure):

    angle_start = FloatParameter("Start angle", units="deg", minimum=0, maximum=360, default=0)
    angle_stop  = FloatParameter("Stop angle", units="deg", minimum=0, maximum=360, default=360)
    angle_step  = FloatParameter("Angle step", units="deg", minimum=0.1, maximum=90, default=3)

    def startup(self):
        self.lia = SR850("GPIB::8")
        self.ps1 = KepcoPS("GPIB::1")
        self.ps2 = BrukerPS("COM3")

        # self.lia.harmonic = 2
        # self.ps1.output = True - Output already enabled in KepcoPS startup
        # self.ps2.output = True - Output already enabled in BrukerPS startup
        return

    def execute(self):
        for angle in np.arange(self.angle_start, self.angle_stop + self.angle_step, self.angle_step):
            print('Angle', angle)
            self.stage.angle = angle
            time.sleep(0.2)

            V1w = self.lia.read_1w()
            V2w = self.lia.read_2w()

            data = {
                "angle": angle,
                "V1w": V1w,
                "V2w": V2w
            }
            self.emit("results", data)

            if self.should_stop():
                break

    def shutdown(self):
        self.ps1.output = False
        self.ps2.output = False

    def tesla_to_current(self):
        return
